# An LLM Reading a Page [Agent Patterns - Module 10]

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 extracted a table with a loop, because the table had a fixed
shape. Real pages are rarely that kind. Facts hide in prose, layouts differ
between pages, and the field you want is called "Delivery" here and
"Shipping" there.

That is where the model goes: **Playwright fetches, the LLM interprets.**

### What you will learn

1. The fetch -> reduce -> extract pipeline.
2. Asking Groq for strict JSON and validating what comes back.
3. Why you always pin a schema instead of saying "extract the details".
4. Extracting facts from prose, where selectors cannot reach.
5. Measuring the token cost of the page you just sent.

### Key takeaways

- Reduce before you extract. Never send the whole page if a section will do.
- Declare the schema in the prompt AND check it in Python.
- The LLM is the *parser*, not the *fetcher*. Keep those jobs separate.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

HERE = Path.cwd().resolve()                  # the module folder
FIXTURES = HERE / "fixtures"
FIXTURES.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Module dir : {HERE}")
print(f"Model      : {MODEL} (via Groq)")


### Running Playwright inside Jupyter (Windows)


In [ ]:
# Two environment quirks, handled once here and reused in every notebook:
#
# 1. Jupyter's kernel already runs an asyncio event loop in the main thread.
#    Playwright's SYNC api refuses to start inside a running loop, so we run
#    every browser job in a short-lived worker thread.
# 2. ipykernel installs the Selector event-loop policy on Windows, and that
#    policy cannot spawn subprocesses - which is exactly what launching
#    Chromium needs. We restore the Proactor policy so new loops can.
#
# In a plain .py script neither applies: `with sync_playwright() as p:` just works.

import asyncio
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

def browse(url, job, headless=True):
    """Open `url` in Chromium, hand the Page to job(page), return its result."""
    def _run():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=headless)
            page = browser.new_page()
            page.goto(url)
            try:
                return job(page)
            finally:
                browser.close()          # never leak a Chromium process
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_run).result()

print("browse() ready - policy:", type(asyncio.get_event_loop_policy()).__name__)


### One small Groq client, with 429 backoff


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def ask(prompt, system="You are a precise assistant.", max_tokens=600, temperature=0.0):
    """One chat completion, with exponential backoff for the free tier."""
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
            time.sleep(wait)
    raise RuntimeError("Groq call failed after 5 attempts")

print("Groq helper ready.")


### Write the local fixture page


In [ ]:
# Everything in this module runs against files on disk. No network, ever.
# That makes the notebook deterministic AND safe to re-run offline.

CATALOG = FIXTURES / "catalog.html"
CATALOG.write_text("""<!doctype html>
<html><head><meta charset="utf-8"><title>Northwind Parts - Catalog</title></head>
<body>
  <h1>Northwind Parts</h1>
  <p class="tagline">Industrial fittings, shipped same day.</p>

  <table id="products">
    <tr><th>SKU</th><th>Name</th><th>Price</th><th>Stock</th></tr>
    <tr><td>NW-1001</td><td>Brass elbow 15mm</td><td>3.40</td><td>184</td></tr>
    <tr><td>NW-1002</td><td>Copper tee 22mm</td><td>5.95</td><td>0</td></tr>
    <tr><td>NW-1007</td><td>PTFE tape 12m</td><td>0.85</td><td>1620</td></tr>
    <tr><td>NW-1044</td><td>Stainless clamp 40mm</td><td>2.10</td><td>37</td></tr>
    <tr><td>NW-2100</td><td>Compression valve 15mm</td><td>11.25</td><td>6</td></tr>
  </table>

  <h2>Delivery</h2>
  <p>Orders placed before 14:00 ship the same working day.
     Free delivery over 50 GBP.</p>

  <footer><a href="contact.html" id="contact-link">Contact us</a></footer>
</body></html>
""", encoding="utf-8")

print("wrote", CATALOG)
print("size:", CATALOG.stat().st_size, "bytes")


### 1. Fetch -> reduce -> extract

Three separate steps, deliberately:

1. **Fetch** (Playwright): get the page, produce rendered text.
2. **Reduce** (plain Python): cut it down to the part that matters.
3. **Extract** (LLM): turn that text into a typed object.

Beginners collapse 1 and 3 ("give the agent a browse tool and let it figure
it out"). Keeping them apart is what makes the thing debuggable: when the
output is wrong you can see immediately whether the *fetch* was wrong or the
*interpretation* was.

### Step 1: fetch


In [ ]:
page_text = browse(CATALOG.as_uri(), lambda page: page.inner_text("body"))

print(f"{len(page_text)} chars fetched\n")
print(page_text[:300])


### Step 2: reduce


In [ ]:
# A crude but effective reducer: keep only lines near a keyword. On a real
# page this is the difference between a 400-token prompt and a 40,000-token
# one - which on a free tier is the difference between working and 429.

def reduce_text(text, keyword, window=4):
    lines = [l.rstrip() for l in text.splitlines()]
    hits = [i for i, l in enumerate(lines) if keyword.lower() in l.lower()]
    keep = set()
    for h in hits:
        keep.update(range(max(0, h - window), min(len(lines), h + window + 1)))
    return "\n".join(lines[i] for i in sorted(keep) if lines[i].strip())

delivery = reduce_text(page_text, "delivery")
print("=== reduced to the delivery section ===")
print(delivery)
print()
print(f"{len(page_text)} chars -> {len(delivery)} chars "
      f"({100*len(delivery)/len(page_text):.0f}%)")


### 2. Extract with a pinned schema

The prompt below does three things that "extract the details" does not:

- names **every** field the caller expects,
- states the type of each,
- says what to do when a field is absent (`null`, not invention).

Then Python validates it. A model that returns JSON is not the same as a
model that returned *your* JSON.

### Step 3: extract structured data


In [ ]:
SCHEMA_PROMPT = """Extract the shipping policy from the page text below.

Return ONLY a JSON object, no prose, no markdown fence, with exactly these keys:
  "same_day_cutoff"      : string, the cutoff time as written, or null
  "free_delivery_over"   : number, the order value for free delivery, or null
  "currency"             : string, a 3-letter code, or null

PAGE TEXT:
---
{page}
---"""

raw = ask(SCHEMA_PROMPT.format(page=delivery),
          system="You extract structured data. You output JSON and nothing else.")
print("raw model output:\n", raw)


### Parse and validate


In [ ]:
# Models sometimes wrap JSON in a fenced block even when told not to.
# Strip defensively, then validate the keys - never trust the shape.

def parse_json(text):
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if fence:
        text = fence.group(1).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"no JSON object found in: {text[:200]}")
    return json.loads(text[start:end + 1])

EXPECTED = {"same_day_cutoff", "free_delivery_over", "currency"}

policy = parse_json(raw)
print("parsed:", policy)
print()
missing = EXPECTED - set(policy)
extra = set(policy) - EXPECTED
print("missing keys:", missing or "none")
print("unexpected keys:", extra or "none")
assert not missing, "the model dropped required fields - this is a bug, not a warning"
print("\nschema OK")


Note the `assert`. In a pipeline, a missing key must be a **loud failure**,
not a `.get()` that quietly returns `None` and corrupts everything
downstream. Validation at the boundary is the whole discipline.

### 3. Where the model actually beats a selector

The delivery policy above is prose: "Orders placed before 14:00 ship the
same working day. Free delivery over 50 GBP." There is no
`<span id="cutoff">`. A selector cannot get it. The model can.

Now compare on a task where a selector *would* work - the product table -
and be honest about the trade.

### Same page, structured region: LLM vs selector


In [ ]:
table_text = reduce_text(page_text, "NW-", window=1)

# Note the wrapper key. Asking for a bare JSON ARRAY is a classic trap: models
# routinely return the elements without the enclosing [ ]. Always ask for an
# OBJECT with a named list inside - it parses reliably.
TABLE_PROMPT = """From the product table text below, return ONLY a JSON object
of the form {{"products": [ ... ]}}.

Each element of "products" must have exactly: "sku" (string), "name" (string),
"price_gbp" (number), "in_stock" (boolean, false when stock is 0).

TEXT:
---
{t}
---"""

t0 = time.time()
raw_tab = ask(TABLE_PROMPT.format(t=table_text),
              system="You extract structured data. Output JSON only.", max_tokens=900)
llm_seconds = time.time() - t0

items = parse_json(raw_tab)["products"]

for it in items:
    print(f"  {it['sku']:8s} {it['name'][:26]:28s} {it['price_gbp']:>6}  "
          f"{'in stock' if it['in_stock'] else 'OUT'}")
print(f"\nLLM extraction took {llm_seconds:.1f}s for {len(items)} rows.")


### The selector version, for comparison


In [ ]:
def scrape(page):
    rows = page.locator("#products tr")
    out = []
    for i in range(1, rows.count()):
        c = [x.strip() for x in rows.nth(i).inner_text().split("\t")]
        out.append({"sku": c[0], "name": c[1],
                    "price_gbp": float(c[2]), "in_stock": int(c[3]) > 0})
    return out

t0 = time.time()
sel_items = browse(CATALOG.as_uri(), scrape)
sel_seconds = time.time() - t0

print(f"selector extraction took {sel_seconds:.1f}s for {len(sel_items)} rows")
print(f"LLM was {llm_seconds/sel_seconds:.1f}x slower, and cost tokens.\n")

same = sel_items == items
print("identical results:", same)
if not same:
    print("differences:")
    for a, b in zip(sel_items, items):
        if a != b:
            print("  selector:", a)
            print("  llm     :", b)


### The honest conclusion

On a structured table the selector wins on every axis: faster, free, exact,
and it fails loudly when the page changes instead of hallucinating a
plausible row. Use the LLM where the *structure* is the problem - prose,
irregular layouts, pages you have never seen - and use selectors everywhere
else.

A good browser agent is mostly deterministic code with a small model-shaped
hole in the middle.

### Token budgeting


In [ ]:
# Rough rule: ~4 characters per token for English.

def approx_tokens(s):
    return len(s) // 4

print(f"{'full page text':24s} ~{approx_tokens(page_text):>6} tokens")
print(f"{'reduced (delivery)':24s} ~{approx_tokens(delivery):>6} tokens")
print(f"{'reduced (table)':24s} ~{approx_tokens(table_text):>6} tokens")
print(f"{'raw HTML':24s} ~{approx_tokens(CATALOG.read_text(encoding='utf-8')):>6} tokens")
print()
print("Groq free tier: 8000 tokens/minute.")
print("A real e-commerce page as raw HTML is often 40,000+ tokens - one page")
print("would exceed the minute budget on its own. Reduction is not an")
print("optimisation here, it is a requirement.")


### Pitfalls recap

- **Sending the whole page.** Reduce first. Always.
- **"Extract the details".** Pin every key and its type, and say what null
  means.
- **Trusting the JSON.** Strip fences, validate keys, assert loudly.
- **Using the LLM where a selector works.** Slower, costlier, less accurate.
- **Assuming stable output.** Set `temperature=0` for extraction.

### Next

Notebook 03 lets the model *act*: choosing values and filling a form.